# Milestone 2 Exploration  (Internal Only)

This notebook is to confirm our .py script outputs, not for submission.

In [96]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [97]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [98]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   parent_asin                    20000 non-null  str    
 1   product_title                  20000 non-null  str    
 2   features                       20000 non-null  object 
 3   description                    20000 non-null  object 
 4   categories                     20000 non-null  object 
 5   details                        20000 non-null  object 
 6   price                          11212 non-null  float64
 7   derived_avg_rating             8669 non-null   float64
 8   max_helpful_vote               8669 non-null   float64
 9   n_reviews                      20000 non-null  int64  
 10  review_text                    8669 non-null   str    
 11  candidate_review_title         8669 non-null   str    
 12  candidate_review_text          8669 non-null   str    
 1

In [100]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,max_helpful_vote,n_reviews,review_text,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
0,B07FD46NZM,Podoy WB6X486 Microwave Grease Filters Replace...,[The grease filters are made aluminum mesh and...,"[If you have any problem,please feel free to c...","[Appliances, Parts & Accessories, Range Hood P...","[(Brand Name, ""Podoy""), (Item Weight, ""3.2 oun...",8.77,3.250000,0.0,4,Availability: Replacement filters This worked ...,This worked in my Sears microwave fan just fine,I lost one of my filters behind the stove when...,0.0
1,B07H7YQZGR,K&J 2-Pack Universal Descaling Solution - USA ...,"[âº NATURAL, ODORLESS CLEANING ACTION - K&J's ...",[Hey! You are wise to be contemplating K&J as ...,"[Small Appliance Parts & Accessories, Coffee &...","[(Package Dimensions, ""6.7 x 4.8 x 2.2 inches""...",22.29,4.615385,36.0,65,Easy to use: I used this for descaling my Nesp...,This worked well.,I noticed my coffee stopped tasting as good. T...,36.0
2,B0BVFQY9HH,Waterdrop DA29-00020B NSF 53&42 Certified Refr...,[[Compatible with multiple models] DA29-00020B...,[],"[Appliances, Parts & Accessories, Refrigerator...","[(Product Dimensions, ""9.13\""D x 2.17\""W x 2.1...",19.59,4.530000,145.0,400,Works same as brand name: We were shocked at t...,Excellent value.,Compared to the Samsung filters which retail f...,145.0
3,B01E9H4SZM,Refresh Replacement Universal Fit Charcoal Wat...,[REFRESH YOUR COFFEE MACHINE WATER FILTER with...,"[Why Refresh?, Because quality and value are o...","[Small Appliance Parts & Accessories, Coffee &...","[(Package Dimensions, ""5.9 x 4 x 1.8 inches""),...",8.99,4.593750,7.0,96,Good value: Great value for this multi pack as...,Money well spent,I needed new filters for my Kuerig coffee make...,7.0
4,B0799Q45TT,"BLACK+DECKER Small Portable Washer, Washing Ma...",[5 CYCLE SELECTION—Wash your laundry with this...,"[When it comes to washing machines, this outst...","[Appliances, Laundry Appliances, Washers & Dry...","[(Brand Name, ""BLACK+DECKER""), (Model Info, ""B...",245.55,4.170213,81.0,47,this machine is a mighty one: I was concerned ...,this machine is a mighty one,I was concerned about ordering this considerin...,81.0


In [101]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
derived_avg_rating,11331,0.56655
max_helpful_vote,11331,0.56655
review_text,11331,0.56655
candidate_review_title,11331,0.56655
candidate_review_text,11331,0.56655
candidate_review_helpful_vote,11331,0.56655
price,8788,0.43940
parent_asin,0,0.00000
product_title,0,0.00000
features,0,0.00000


In [102]:
df.describe()

,price,derived_avg_rating,max_helpful_vote,n_reviews,candidate_review_helpful_vote
count,11212.000000,8669.000000,8669.000000,20000.00000,8669.000000
mean,88.479685,4.256767,5.083977,3.53030,5.083977
std,310.481798,1.092489,38.984341,23.69879,38.984341
min,0.010000,1.000000,0.000000,0.00000,0.000000
25%,14.990000,4.000000,0.000000,0.00000,0.000000
50%,26.990000,4.750000,0.000000,0.00000,0.000000
75%,59.950000,5.000000,2.000000,1.00000,2.000000
max,7691.010000,5.000000,1835.000000,1220.00000,1835.000000


## Debugging Code

The code below were explorations to help inform the pipeline and find bugs. Codex was used to generate quick debugging code below.

In [103]:
def present(s):
    return s.notna() & s.astype(str).str.strip().ne("")

df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

summary = pd.crosstab(
    index=[
        df["has_n_reviews"],
        df["has_avg_rating"],
        df["has_candidate_title"],
        df["has_candidate_text"],
    ],
    columns="count",
).reset_index()

summary = summary.sort_values("count", ascending=False)
summary

col_0,has_n_reviews,has_avg_rating,has_candidate_title,has_candidate_text,count
0,False,False,False,False,11331
2,True,True,True,True,8668
1,True,True,True,False,1


In [104]:
# Review count exists, but no candidate review shown
df.loc[
    (df["n_reviews"] > 0)
    & (~present(df["candidate_review_title"]))
    & (~present(df["candidate_review_text"])),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
].head(20)

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote


In [105]:
def present(s):
    cleaned = s.astype("string").str.strip()
    missing_tokens = {"", "na", "n/a", "nan", "none", "null"}
    return cleaned.notna() & ~cleaned.str.lower().isin(missing_tokens)

In [106]:
df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

In [107]:
df.loc[
    df["product_title"].str.contains(
        "Pour Over Coffee Dripper Stainless Steel Double Layer Mesh",
        regex=False,
        na=False,
    ),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
911,B07D6L2QX2,Pour Over Coffee Dripper Stainless Steel Doubl...,0,NaN,NaN,NaN,NaN


In [108]:
df.loc[
    df["parent_asin"] == "B09VT3BS1G",
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
3476,B09VT3BS1G,"MoMoSun Furniture Dolly,Extendable Washing Mac...",0,NaN,NaN,NaN,NaN
